# Student Ranker Model Training Notebook
This notebook trains a lightweight tabular student ranker model (`GradientBoostingRegressor`) using pseudo-labels generated by the heavier Cross-Encoder model. This runs in milliseconds on CPU, satisfying the 5-minute hackathon constraint.

In [ ]:
# 1. Install dependencies
!pip install sentence-transformers scikit-learn pandas numpy torch

In [ ]:
# 2. Check GPU availability (highly recommended to speed up Cross-Encoder labeling)
import torch
print('GPU/CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Using Device:', torch.cuda.get_device_name(0))

# Note on Uploading Data
Before running the training script below, make sure to upload the following files from your local repository to this Colab instance:
1. `candidates.jsonl` (or a sample of it)
2. `embeddings_full.pkl` (precomputed embeddings mapping candidate_id to BGE vector)
3. `job_description_full_extracted.txt`
4. `train_student_ranker.py` (the python training script)

You can upload them by clicking the Folder icon on the left sidebar in Colab and selecting the files.

In [ ]:
# 3. Run training pipeline
# This will select the top 10,000 candidates retrieved by BM25F + BGE,
# score them using the local Cross-Encoder (Teacher), and train the tabular regressor (Student).
# On a Colab T4 GPU, this runs in under 2 minutes.
!python train_student_ranker.py --candidates ./candidates.jsonl --jd ./job_description_full_extracted.txt --embeddings ./embeddings_full.pkl --output ./student_ranker.pkl --size 10000

In [ ]:
# 4. Verify model exists and download it
import os
if os.path.exists('student_ranker.pkl'):
    print('Model trained and saved successfully!')
    try:
        from google.colab import files
        files.download('student_ranker.pkl')
    except ImportError:
        print('Not in Google Colab, download manually.')
else:
    print('Error: student_ranker.pkl not found.')